# CIFAR-10 Neural Network Assignment

## 1. Import Libraries and Setup

## QUICK START
**Output Files:**
- experiment_a_depth.png
- experiment_b_lr.png  
- experiment_c_bs.png

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import time

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BATCH_SIZE = 128
LEARNING_RATE = 0.1
EPOCHS = 20
SUBSET_SIZE = 1.0

## 2. Load and Prepare CIFAR-10 Dataset

In [ ]:
def load_data_cifar10(subset_size=1.0):
    """Load CIFAR-10 using torchvision"""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )
    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )
    
    # Apply subset if needed
    train_size = int(len(trainset) * subset_size)
    test_size = int(len(testset) * subset_size)
    
    trainset, _ = random_split(
        trainset, [train_size, len(trainset) - train_size],
        generator=torch.Generator().manual_seed(0)
    )
    testset, _ = random_split(
        testset, [test_size, len(testset) - test_size],
        generator=torch.Generator().manual_seed(0)
    )
    
    # Validation split
    val_size = int(len(trainset) * 0.2)
    train_ds, val_ds = random_split(
        trainset, [len(trainset) - val_size, val_size],
        generator=torch.Generator().manual_seed(0)
    )
    
    print(f"Training data shape: {len(train_ds)}, Labels shape: ({len(train_ds)},)")
    print(f"Validation data shape: {len(val_ds)}, Labels shape: ({len(val_ds)},)")
    print(f"Test data shape: {len(testset)}, Labels shape: ({len(testset)},)")
    print(f"Number of classes: 10")
    
    return train_ds, val_ds, testset

def load_batches(train_ds, val_ds, test_ds, batch_size=128):
    """Create DataLoaders"""
    train_iter = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_iter = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_iter = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_iter, val_iter, test_iter

# Load data
train_ds, val_ds, test_ds = load_data_cifar10(SUBSET_SIZE)
train_iter, val_iter, test_iter = load_batches(train_ds, val_ds, test_ds, BATCH_SIZE)

In [ ]:
# ==================== FILE SAVING SETUP FOR COLAB ====================
import os

# Check if running on Colab
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/gdrive')
    SAVE_PATH = '/content/gdrive/My Drive/CIFAR10_Results/'
    print("✓ Running on Google Colab - saving to Google Drive")
except:
    IN_COLAB = False
    SAVE_PATH = './'
    print("✓ Running locally - saving to current directory")

# Create save directory if it doesn't exist
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"Files will be saved to: {SAVE_PATH}\n")

---

## EXPERIMENT MENU

Choose one experiment to run:

| Option | Experiment | Time |
|--------|-----------|------|
| 1 | Softmax Training | 4 min |
| 2 | CNN Depth Analysis | 15 min |
| 3 | Learning Rate Analysis | 12 min |
| 4 | Batch Size Study | 12 min |

Scroll down to your chosen section and run those cells only.

## 3. Define Neural Network Models

In [ ]:
class Softmax(nn.Module):
    """Softmax Regression"""
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(3 * 32 * 32, 10)
    
    def forward(self, x):
        return self.fc(x.view(x.size(0), -1))

def create_cnn(depth=2):
    """Create CNN with variable depth"""
    layers = []
    in_ch, out_ch = 3, 32
    pool_count = 0
    max_pools = 3
    
    for i in range(depth):
        layers.append(nn.Conv2d(in_ch, out_ch, 3, padding=1))
        layers.append(nn.ReLU())
        
        should_pool = (
            (i + 1) % 2 == 0 and 
            pool_count < max_pools
        )
        
        if should_pool:
            layers.append(nn.MaxPool2d(2, 2))
            pool_count += 1
        
        in_ch = out_ch
        if i % 4 == 1:
            out_ch = min(out_ch * 2, 256)
    
    layers.append(nn.AdaptiveAvgPool2d(1))
    
    model = nn.Sequential(*layers)
    return model

class CNN(nn.Module):
    """CNN with variable depth"""
    def __init__(self, depth=2):
        super().__init__()
        self.conv = create_cnn(depth)
        self.conv = self.conv.to(device)
        
        with torch.no_grad():
            x = torch.randn(1, 3, 32, 32).to(device)
            x = self.conv(x)
            flat_size = x.view(1, -1).size(1)
        
        self.fc = nn.Sequential(
            nn.Linear(flat_size, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

class SimpleCNN(nn.Module):
    """Simple 2-layer CNN"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.features(x)
        return self.fc(x.view(x.size(0), -1))

In [ ]:


## OPTIONAL: Diagnostics
Skip this section unless debugging

## 4. Training and Evaluation Functions

## PART 1: Softmax Training (5 marks)
Run cell below to train Softmax model

In [ ]:
print("\n" + "="*80)
print("PART 1: SOFTMAX TRAINING")
print("="*80)

softmax_net = Softmax().to(device)
softmax_result = train(softmax_net, train_iter, val_iter, test_iter, EPOCHS, LEARNING_RATE)

print(f"\n[Results]")
print(f"{'Metric':<20} {'Value':<15}")
print("-"*35)
print(f"{'Test Accuracy':<20} {softmax_result['test_acc']:.4f}")
print(f"{'Train Loss':<20} {softmax_result['train_losses'][-1]:.4f}")
print(f"{'Val Loss':<20} {softmax_result['val_losses'][-1]:.4f}")
print(f"{'Time (s)':<20} {softmax_result['train_time']:.0f}")

print(f"\n[Errors by Epoch]")
print(f"{'Epoch':<10} {'Train Loss':<15} {'Val Loss':<15} {'Val Acc':<15}")
print("-"*55)
for epoch in range(len(softmax_result['train_losses'])):
    print(f"{epoch+1:<10} {softmax_result['train_losses'][epoch]:<15.4f} {softmax_result['val_losses'][epoch]:<15.4f} {softmax_result['val_accs'][epoch]:<15.4f}")

In [ ]:
def accuracy(y_hat, y):
    """Compute accuracy"""
    return float((torch.argmax(y_hat, dim=1) == y).sum()) / len(y)

def train_epoch(net, train_iter, loss_fn, optimizer):
    """Train one epoch"""
    net.train()
    total_loss = 0
    num_batches = 0
    for X, y in train_iter:
        X, y = X.to(device), y.to(device)
        y_hat = net(X)
        loss = loss_fn(y_hat, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches

def evaluate(net, data_iter, loss_fn):
    """Evaluate model"""
    net.eval()
    total_loss = 0
    total_acc = 0
    num_batches = 0
    
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            loss = loss_fn(y_hat, y)
            
            total_loss += loss.item()
            total_acc += accuracy(y_hat, y)
            num_batches += 1
    
    return total_loss / num_batches, total_acc / num_batches

def train(net, train_iter, val_iter, test_iter, epochs, lr, weight_decay=0):
    """Train model"""
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, weight_decay=weight_decay)
    
    train_losses, val_losses, val_accs = [], [], []
    start_time = time.time()
    
    for epoch in range(epochs):
        train_loss = train_epoch(net, train_iter, loss_fn, optimizer)
        val_loss, val_acc = evaluate(net, val_iter, loss_fn)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:2d}/{epochs} | Loss: {train_loss:.3f} | Val: {val_loss:.3f} | Acc: {val_acc:.3f}")
    
    elapsed = time.time() - start_time
    test_loss, test_acc = evaluate(net, test_iter, loss_fn)
    print(f"  Test Acc: {test_acc:.4f} | Time: {elapsed:.0f}s\n")
    
    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'test_acc': test_acc,
        'train_time': elapsed
    }

---
## PART 2A: CNN Depth Analysis (5 marks)
Time: ~15 minutes

Quick LR Test (5 epochs)

In [ ]:
print("\n" + "="*60)
print("Quick test: Finding best learning rate")
print("="*60)

quick_lrs = [0.001, 0.01, 0.05, 0.1, 0.2]
quick_results = {}

for lr in quick_lrs:
    print(f"Testing LR={lr}...")
    net = SimpleCNN().to(device)
    result = train(net, train_iter, val_iter, test_iter, 5, lr)
    quick_results[lr] = result['test_acc']

best_lr = max(quick_results, key=quick_results.get)
print(f"\nBest LR: {best_lr} (Accuracy: {quick_results[best_lr]:.4f})")

In [ ]:
print("\n" + "="*60)
print("Sanity check: Model overfitting on single batch")
print("="*60)

# Get one batch
X_batch, y_batch = next(iter(train_iter))
X_batch, y_batch = X_batch[:32].to(device), y_batch[:32].to(device)  # Use 32 samples

# Create fresh model
debug_net = SimpleCNN().to(device)
debug_loss_fn = nn.CrossEntropyLoss()
debug_opt = optim.SGD(debug_net.parameters(), lr=0.1)  # Higher LR for fast learning

print(f"Testing on batch: {X_batch.shape}, {y_batch.shape}")

# Try to overfit for 50 steps
losses = []
accs = []
for step in range(50):
    debug_net.train()
    out = debug_net(X_batch)
    loss = debug_loss_fn(out, y_batch)
    
    debug_opt.zero_grad()
    loss.backward()
    debug_opt.step()
    
    losses.append(loss.item())
    acc = accuracy(out, y_batch)
    accs.append(acc)
    
    if (step + 1) % 10 == 0:
        print(f"  Step {step+1}: Loss {loss.item():.4f}, Acc {acc:.4f}")

print(f"\nResult: Loss {losses[0]:.4f} → {losses[-1]:.4f}")
print(f"        Acc  {accs[0]:.4f} → {accs[-1]:.4f}")

if accs[-1] > 0.8:
    print("Model can learn - architecture is OK")
else:
    print("Model cannot overfit - check architecture")

print()

In [ ]:
print("\n" + "="*80)
print("PART 2A: CNN DEPTH ANALYSIS")
print("="*80)

depths = [2, 8, 16, 32]
results_depth = {}

for depth in depths:
    print(f"Training depth {depth}...")
    net = CNN(depth=depth).to(device)
    result = train(net, train_iter, val_iter, test_iter, EPOCHS, LEARNING_RATE)
    results_depth[depth] = result

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0, 0]
for d in depths:
    ax.plot(results_depth[d]['train_losses'], label=f'Depth {d}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for d in depths:
    ax.plot(results_depth[d]['val_losses'], label=f'Depth {d}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for d in depths:
    ax.plot(results_depth[d]['val_accs'], label=f'Depth {d}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
test_accs = [results_depth[d]['test_acc'] for d in depths]
ax.bar([str(d) for d in depths], test_accs, color='steelblue', alpha=0.7)
ax.set_ylabel('Test Accuracy')
ax.set_xlabel('Depth')
ax.set_title('Final Test Accuracy')
ax.grid(True, axis='y', alpha=0.3)
for i, acc in enumerate(test_accs):
    ax.text(i, acc + 0.01, f'{acc:.3f}', ha='center')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'experiment_a_depth.png'), dpi=100, bbox_inches='tight')
plt.show()
print("Saved: experiment_a_depth.png")

print(f"\n[Results]")
print(f"{'Depth':<8} {'Test Acc':<10} {'Time (s)':<10}")
print("-"*28)
for d in depths:
    print(f"{d:<8} {results_depth[d]['test_acc']:.4f}     {results_depth[d]['train_time']:.0f}")

---
## PART 2B: Learning Rate Analysis (3 marks)
Time: ~12 minutes
Run Setup cells first

In [ ]:
print("\n" + "="*80)
print("PART 2B: LEARNING RATE ANALYSIS")
print("="*80)

learning_rates = [0.000001, 0.0001, 0.001, 0.01, 1.0]
results_lr = {}

for lr in learning_rates:
    print(f"Training LR={lr}...")
    net = SimpleCNN().to(device)
    result = train(net, train_iter, val_iter, test_iter, EPOCHS, lr)
    results_lr[lr] = result

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0, 0]
for lr in learning_rates:
    ax.plot(results_lr[lr]['train_losses'], label=f'LR={lr}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for lr in learning_rates:
    ax.plot(results_lr[lr]['val_losses'], label=f'LR={lr}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for lr in learning_rates:
    ax.plot(results_lr[lr]['val_accs'], label=f'LR={lr}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
test_accs = [results_lr[lr]['test_acc'] for lr in learning_rates]
ax.bar([f'{lr:.6f}' for lr in learning_rates], test_accs, color='coral', alpha=0.7)
ax.set_ylabel('Test Accuracy')
ax.set_xlabel('Learning Rate')
ax.set_title('Final Test Accuracy')
ax.grid(True, axis='y', alpha=0.3)
for i, acc in enumerate(test_accs):
    ax.text(i, acc + 0.01, f'{acc:.3f}', ha='center')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'experiment_b_lr.png'), dpi=100, bbox_inches='tight')
plt.show()
print("Saved: experiment_b_lr.png")

print(f"\n[Results]")
print(f"{'LR':<12} {'Test Acc':<10} {'Time (s)':<10}")
print("-"*32)
for lr in learning_rates:
    print(f"{lr:<12.6f} {results_lr[lr]['test_acc']:.4f}     {results_lr[lr]['train_time']:.0f}")


## PART 2C: Batch Size Study (3 marks)
Time: ~12 minutes

In [ ]:
print("\n" + "="*80)
print("PART 2C: BATCH SIZE STUDY")
print("="*80)

batch_sizes = [1, 8, 16, 64, 256]
results_bs = {}

for bs in batch_sizes:
    print(f"Training batch size {bs}...")
    train_iter_bs, val_iter_bs, test_iter_bs = load_batches(train_ds, val_ds, test_ds, batch_size=bs)
    net = SimpleCNN().to(device)
    result = train(net, train_iter_bs, val_iter_bs, test_iter_bs, EPOCHS, LEARNING_RATE)
    results_bs[bs] = result

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0, 0]
for bs in batch_sizes:
    ax.plot(results_bs[bs]['train_losses'], label=f'BS={bs}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for bs in batch_sizes:
    ax.plot(results_bs[bs]['val_losses'], label=f'BS={bs}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for bs in batch_sizes:
    ax.plot(results_bs[bs]['val_accs'], label=f'BS={bs}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
test_accs = [results_bs[bs]['test_acc'] for bs in batch_sizes]
ax.bar([str(bs) for bs in batch_sizes], test_accs, color='lightgreen', alpha=0.7)
ax.set_ylabel('Test Accuracy')
ax.set_xlabel('Batch Size')
ax.set_title('Final Test Accuracy')
ax.grid(True, axis='y', alpha=0.3)
for i, acc in enumerate(test_accs):
    ax.text(i, acc + 0.01, f'{acc:.3f}', ha='center')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'experiment_c_bs.png'), dpi=100, bbox_inches='tight')
plt.show()
print("Saved: experiment_c_bs.png")

print(f"\n[Results]")
print(f"{'Batch Size':<12} {'Test Acc':<10} {'Time (s)':<10}")
print("-"*32)
for bs in batch_sizes:
    print(f"{bs:<12} {results_bs[bs]['test_acc']:.4f}     {results_bs[bs]['train_time']:.0f}")

---
## RESULTS

In [ ]:
print("\n" + "="*80)
print("FILES READY FOR DOWNLOAD")
print("="*80)

import glob
saved_files = glob.glob(os.path.join(SAVE_PATH, '*.png'))
print(f"\nGenerated files ({len(saved_files)}):") 
for f in sorted(saved_files):
    print(f"  {os.path.basename(f)}")

print(f"\nLocation: {SAVE_PATH}")

if IN_COLAB:
    print("\nDownload from Google Colab:")
    print("  Files > My Drive > CIFAR10_Results/")
else:
    print("\nFiles saved locally")
    
print("="*80)